# Importación de datos {#sec-data-import}

## Introducción

En este capítulo vamos a empezar a trabajar con tus propios datos. Aprenderás a leer en Python archivos rectangulares de texto plano. Solo veremos lo más básico de la importación de datos, pero muchos de los principios se aplican a otras formas de datos. Terminaremos con algunas indicaciones para abrir otros tipos de datos.

### Requisitos previos

Necesitarás tener instalado el paquete **pandas**. Debes asegurarte de tener **pandas** instalado. Para comprobarlo, e importar **pandas** en tu sesión, ejecuta

In [ ]:
import pandas as pd

Si este comando falla, no tienes **pandas** instalado. Abre la terminal en Visual Studio Code (Terminal -> New Terminal), haz `cd` a la carpeta en la que estás trabajando y escribe `uv add pandas`. Ten en cuenta que, una vez instalado **pandas**, la convención es importarlo en tu sesión de Python con el nombre `pd` poniendo `import pandas as pd` al principio de tu script.

## Primeros pasos

**pandas** admite una enorme variedad de formatos de entrada y salida: Stata (.dta), Excel (.xls, .xlsx), csv, tsv, formatos de big data (HDF5, parquet), JSON, SAS, SPSS, SQL y más; hay una [lista completa](https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html) de los formatos disponibles en la documentación.

![De la documentación de pandas](https://pandas.pydata.org/pandas-docs/stable/_images/02_io_readwrite.svg)

Aunque **pandas** tiene muchísimas formas de leer datos y cargarlos en tu sesión de Python, aquí nos centraremos en el humilde archivo de tabla en texto plano; por ejemplo, csv (valores separados por comas) y tsv (valores separados por tabulaciones).

### Leer datos de un archivo

Todo lo necesario para abrir archivos de tabla en texto plano está contenido en una sola función, `pd.read_csv()`. Admite numerosos argumentos, pero los dos más importantes son el primero (sin nombre), que indica la ruta a los datos, y `sep=` (un argumento con nombre), que le dice a **pandas** si los valores están separados por comas, tabulaciones u otro carácter; sin embargo, si dejas este campo vacío, **pandas** lo adivinará por ti. Para ver el conjunto completo de argumentos, ejecuta `help(pd.read_csv)`.

Así se ve un archivo CSV sencillo con una fila de nombres de columna (también llamada habitualmente fila de encabezado) y seis filas de datos (usando la terminal):

In [ ]:
! cat data/students.csv

Fíjate en que es un archivo CSV, así que los valores están separados por comas. Ahora carguémoslo en un dataframe de **pandas** en Python:

In [ ]:
students = pd.read_csv("data/students.csv")
students

Si quieres descargar estos datos para probarlo por tu cuenta, ve a [este enlace](https://github.com/aeturrell/python4DS/blob/main/data/students.csv), haz clic derecho en 'Raw' y selecciona "Guardar enlace como...". Guarda los datos en un directorio llamado 'data' con el nombre 'students.csv'. Este directorio debe estar dentro de tu carpeta activa de Visual Studio Code. Puedes comprobar en qué carpeta estás ejecutando el siguiente código:

```python
import os

os.getcwd()  # get current working directory (cwd)
```

Si devuelve 'python4DS', tus datos descargados deberían estar en 'python4DS/data/students.csv'.

El primer argumento de `read_csv()` fue la ruta a los datos, y **pandas** adivinó que este archivo usa comas como separador.

La función de lectura de CSV crea automáticamente un nuevo *índice* (que es simplemente la posición de cada fila) y toma la primera línea de datos como *encabezado* o *nombres de columna*. Pero quizá quieras ajustar este comportamiento de varias formas.

1.  A veces hay algunas líneas de metadatos al principio del archivo. Puedes usar `skiprows=n` para omitir las primeras `n` líneas, por ejemplo `pd.read_csv("data/students.csv", skiprows=2)`.

2.  Puede que los datos no tengan nombres de columna. Puedes usar `names = ` con una lista para indicarle a `read_csv()` que use otra opción para los nombres de columna. Por ejemplo, `pd.read_csv("data/students.csv", names=range(5))` pondría los números del 0 al 4 como nombres de columna.

3. Quizá quieras cambiar qué columna se usa como índice. El comportamiento por defecto es crear un índice, pero en estos datos vemos que ya hay una columna de ID que podríamos usar. Para ello, usa el argumento `index_col=`, por ejemplo `pd.read_csv("data/students.csv", index_col=0)`.

Esto es todo lo que necesitas saber para leer \~75% de los archivos CSV que encontrarás en la práctica. Los archivos separados por tabulaciones y los de ancho fijo se leen con la misma función.

### Primeros pasos

Echemos otro vistazo a los datos de `students`.

Una vez que lees los datos, el primer paso suele consistir en transformarlos de alguna manera para que sea más fácil trabajar con ellos en el resto del análisis. Por ejemplo, los nombres de columna del archivo `students` que leímos tienen formatos poco estándar.

Podrías renombrarlas una por una con `.rename()` o usar una función práctica de otro paquete para limpiarlas y convertirlas todas a snake case de una vez. Usaremos el paquete **skimpy** para ello. Instálalo ejecutando `uv add skimpy` en la terminal.

De **skimpy** usaremos la función `clean_columns()`; recibe un dataframe y devuelve un dataframe con los nombres de las variables convertidos a snake case.

In [ ]:
from skimpy import clean_columns

students = clean_columns(students)
students

Otra tarea habitual después de leer los datos es revisar los tipos de las variables. En la columna `favourite_food` hay varios alimentos y luego el valor `NaN`, que se ha leído como un número de punto flotante en lugar de un string faltante. Podemos solucionarlo convirtiendo explícitamente esa columna a strings:

In [ ]:
students["favourite_food"] = students["favourite_food"].astype("string")
students

De forma similar, `"age"` tiene tipos de datos mezclados: ¡string y entero! Convirtamos 'five' en el número cinco.

In [ ]:
students["age"] = students["age"].replace("five", 5)
students["age"]

En un momento también convertiremos esta en una columna de enteros.

Otro ejemplo en el que el tipo de dato es incorrecto es `meal_type`. Es una variable categórica con un conjunto conocido de valores posibles. **pandas** tiene un tipo de dato especial para estos casos:

In [ ]:
students["meal_plan"] = students["meal_plan"].astype("category")
students["meal_plan"]

Fíjate en que los valores de la variable `meal_type` siguen siendo exactamente los mismos, pero el tipo de la variable ha cambiado de object a category.

Es algo tedioso tener que recorrer las columnas una por una con asignaciones de una línea para aplicar el tipo. Una alternativa es pasar un diccionario que asigne tipos a los nombres de columna, como sigue:

In [ ]:
students = students.astype({"student_id": "int", "full_name": "string", "age": "int"})
students.info()

### Ejercicios

1.  ¿Qué función usarías para leer un archivo cuyos campos estuvieran separados por "\|"?

## Leer datos de varios archivos

A veces tus datos están repartidos en varios archivos en lugar de estar en uno solo. Por ejemplo, podrías tener datos de ventas de varios meses, con los datos de cada mes en un archivo distinto: `01-sales.csv` para enero, `02-sales.csv` para febrero y `03-sales.csv` para marzo.

Con `pd.read_csv()` puedes leer estos datos uno por uno y luego apilarlos en un único dataframe usando la función `pd.concat()`. Se ve así:


In [ ]:
list_of_dataframes = [
    pd.read_csv(x)
    for x in ["data/01-sales.csv", "data/02-sales.csv", "data/03-sales.csv"]
]
sales_files = pd.concat(list_of_dataframes)
sales_files

Si tienes muchos archivos que leer, puede resultar engorroso escribir sus nombres en una lista. En su lugar, puedes usar el paquete **glob** (incluido en Python) para que encuentre los archivos por ti buscando un patrón en sus nombres. Ten en cuenta que puede haber otros archivos CSV en el directorio data/, así que aquí especificamos `"*-sales.csv"` para asegurarnos de obtener solo los archivos que incluyen la palabra sales. Aquí, `"*"` actúa como comodín: representa cualquier secuencia de caracteres.

In [ ]:
import glob

list_of_csvs = glob.glob("data/*-sales.csv")
print("List of csvs is:")
print(list_of_csvs, "\n")
sales_files = pd.concat([pd.read_csv(x) for x in list_of_csvs])
sales_files

## Escribir en un archivo

Así como el patrón típico para leer archivos es `pd.read_FILETYPE()`, donde el tipo de archivo puede ser, por ejemplo, CSV, todas las formas de escribir dataframes de **pandas** en disco siguen el patrón `DATAFRAME.to_FILETYPE()`. Así que, para escribir nuestros datos de ventas en un archivo CSV, el código será `sales_files.to_csv(FILEPATH)`, donde la ruta es la ubicación más el nombre del archivo en el que quieres escribir.

Veamos un ejemplo de cómo escribir datos en un archivo usando nuestros datos de estudiantes, en los que ya hicimos un buen trabajo ajustando los tipos de datos:

In [ ]:
students.to_csv("data/students-clean.csv")

Ahora volvamos a leerlo y revisemos la información sobre los tipos de datos:

In [ ]:
pd.read_csv("data/students-clean.csv").info()

¿Notas algo? ¡Perdimos gran parte del buen trabajo que hicimos con los tipos de datos! Aunque **pandas** adivinó que algunas columnas son enteros, perdimos las variables string y categóricas. La razón es que los archivos de texto plano no pueden guardar información contextual (aunque **pandas** adivinará los tipos de datos de algunas columnas).

Si quieres guardar datos en un archivo y que recuerde los tipos de datos, tendrás que usar otro formato. Para almacenamiento temporal, recomendamos el formato *feather*, ya que es muy rápido e interoperable con otros lenguajes de programación. La interoperabilidad es una buena razón para evitar formatos de archivo específicos de un lenguaje, como .dta de Stata, .rds de R y .pickle de Python.

Ten en cuenta que el formato feather tiene una dependencia adicional: un paquete llamado **pyarrow**. Para instalarlo, ejecuta `uv add pyarrow` en una ventana de terminal.

Este es un ejemplo de cómo escribir en un archivo feather:

In [ ]:
students.to_feather("data/students-clean.feather")

Ahora volvamos a abrir ese archivo feather y veamos la información asociada.

In [ ]:
pd.read_feather("data/students-clean.feather").info()

Guardar en este formato conservó la información de los tipos de datos.

### Leer y escribir otros formatos de datos

La imagen del principio de este capítulo te da una idea de qué otros formatos están disponibles, pero puedes encontrar una lista completa en la [documentación oficial de **pandas** sobre entrada y salida](https://pandas.pydata.org/docs/user_guide/io.html).

In [ ]:
import os

os.remove("data/students-clean.csv")
os.remove("data/students-clean.feather")